# LeetCode 2206 — Divide Array Into Equal Pairs bằng Python

Link đề: https://leetcode.com/problems/divide-array-into-equal-pairs/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Bit Manipulation, Counting

Skeleton LeetCode:

```python
class Solution:
    def divideArray(self, nums: List[int]) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chia `2n` số thành `n` **cặp**, mỗi cặp gồm **hai số bằng nhau**, mỗi phần tử dùng đúng một lần.
2. Chuyển đề về điều kiện đếm: **mỗi giá trị xuất hiện một số chẵn lần**.
3. Biết dùng `Counter`, `set` (kỹ thuật bật/tắt), sắp xếp + so cặp kề nhau, và mảng đếm có chỉ số theo giá trị.
4. Hiểu phép **parity (chẵn/lẻ)** `% 2` và `^` (XOR) trong bối cảnh ghép cặp.
5. So sánh 5 cách, trace trạng thái, test ngẫu nhiên và tránh lỗi như chỉ kiểm tra độ dài hoặc quên chia hết cho 2.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` có `2 * n` phần tử. Hãy chia `nums` thành `n` **cặp** sao cho:

- mỗi phần tử thuộc **đúng một** cặp;
- hai phần tử trong mỗi cặp **bằng nhau**.

Trả về `True` nếu chia được, ngược lại `False`.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[3,2,3,2,2,2]` | `True` | có 3 cặp: `(2,2)`, `(3,3)`, `(2,2)` |
| 2 | `[1,2,3,4]` | `False` | không có hai số nào bằng nhau |

Constraints: `nums.length == 2 * n`, `1 <= n <= 500`, `1 <= nums[i] <= 500`.

Quan trọng: độ dài mảng **luôn chẵn** theo đề, nên ta không phải lo trường hợp lẻ phần tử.

In [1]:
examples = [([3, 2, 3, 2, 2, 2], True), ([1, 2, 3, 4], False)]
for nums, expected in examples:
    print(nums, "| mong đợi:", expected, "| độ dài:", len(nums), "| số cặp cần:", len(nums) // 2)

[3, 2, 3, 2, 2, 2] | mong đợi: True | độ dài: 6 | số cặp cần: 3
[1, 2, 3, 4] | mong đợi: False | độ dài: 4 | số cặp cần: 2


## 3. Kiến thức nền

### 3.1. `list.count`, `list.remove` và vòng lặp xóa phần tử

`a.count(x)` đếm số lần `x` xuất hiện. `a.remove(x)` xóa **lần xuất hiện đầu tiên** của `x` (báo `ValueError` nếu không có). Hai hàm này đủ để viết cách "ghép cặp trực tiếp".

In [2]:
a = [3, 2, 3, 2, 2, 2]
print(a.count(2), a.count(3), a.count(9))
b = a[:]
b.remove(2)
print(b)

4 2 0
[3, 3, 2, 2, 2]


### 3.2. `collections.Counter`

`Counter(iterable)` tạo bảng đếm `giá trị → số lần`. Tra cứu giá trị không có trả về `0`, không báo lỗi. `.values()` cho các số lần, `.items()` cho cặp `(giá trị, số lần)`.

In [3]:
from collections import Counter

cnt = Counter([3, 2, 3, 2, 2, 2])
print(cnt)
print(cnt[2], cnt[3], cnt[99])
print(list(cnt.values()), list(cnt.items()))

Counter({2: 4, 3: 2})
4 2 0
[2, 4] [(3, 2), (2, 4)]


### 3.3. Chẵn / lẻ: `% 2`, `all`, `any`

`n % 2 == 0` nghĩa là `n` chẵn. `all(điều_kiện for x in ds)` là `True` khi **mọi** phần tử thỏa; `any(...)` là `True` khi **có ít nhất một** phần tử thỏa. Hai hàm dừng sớm ngay khi biết kết quả.

In [4]:
vals = [4, 2, 6]
print(all(v % 2 == 0 for v in vals))
print(any(v % 2 == 1 for v in [4, 2, 5]))
print([n % 2 for n in range(6)])

True
True
[0, 1, 0, 1, 0, 1]


### 3.4. `set` và kỹ thuật "bật / tắt" (toggle)

Với `set`, `s.add(x)` thêm, `s.remove(x)` xóa (lỗi nếu không có), `x in s` kiểm tra nhanh. Ý tưởng toggle: gặp `x` **lần lẻ** thì thêm vào, gặp **lần chẵn** thì xóa. Cuối cùng set chỉ giữ các giá trị xuất hiện **lẻ** lần.

In [5]:
seen = set()
for x in [3, 2, 3, 2, 2, 2]:
    if x in seen:
        seen.remove(x)
    else:
        seen.add(x)
    print("đọc", x, "-> seen =", seen)

đọc 3 -> seen = {3}
đọc 2 -> seen = {2, 3}
đọc 3 -> seen = {2}
đọc 2 -> seen = set()
đọc 2 -> seen = {2}
đọc 2 -> seen = set()


### 3.5. Sắp xếp và so từng cặp kề nhau

`sorted(nums)` xếp tăng dần, các giá trị bằng nhau đứng cạnh nhau. Khi đó cặp tại vị trí `(0,1), (2,3), (4,5), ...` có thể kiểm tra bằng `range(0, n, 2)` (bước nhảy 2).

In [6]:
arr = sorted([3, 2, 3, 2, 2, 2])
print(arr)
print([(arr[i], arr[i + 1]) for i in range(0, len(arr), 2)])
print(list(range(0, 6, 2)))

[2, 2, 2, 2, 3, 3]
[(2, 2), (2, 2), (3, 3)]
[0, 2, 4]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ghép cặp bằng tay với `[3,2,3,2,2,2]`: lấy `3` đầu tiên → tìm `3` còn lại (vị trí 2) → ghép. Lấy `2` đầu → ghép với một `2` khác. Còn lại hai số `2` → ghép. **Hết số**, nên `True`.

Với `[1,2,3,4]`: lấy `1` mà không còn `1` nào khác → kẹt, `False`.

Nhìn theo **số lần xuất hiện**: giá trị `3` xuất hiện 2 lần, giá trị `2` xuất hiện 4 lần. Cả hai đều **chẵn**. Một giá trị xuất hiện lẻ lần (ví dụ 1 lần) thì luôn thừa một phần tử không ai ghép cùng. Vậy đề tương đương: **mọi giá trị đều xuất hiện số chẵn lần**.

In [7]:
nums = [3, 2, 3, 2, 2, 2]
freq = {}
for x in nums:
    freq[x] = freq.get(x, 0) + 1
print(freq)
print({v: c % 2 == 0 for v, c in freq.items()})
print("chia được:", all(c % 2 == 0 for c in freq.values()))

{3: 2, 2: 4}
{3: True, 2: True}
chia được: True


## 5. Cách 1 — Brute Force / baseline: ghép cặp trực tiếp bằng `remove`

Mô phỏng đúng như tay làm: lặp khi danh sách còn phần tử, lấy phần tử đầu, tìm bạn cặp trong phần còn lại; không tìm thấy → `False`.

In [8]:
def pair_up(nums):
    rest = list(nums)
    pairs = []
    while rest:
        x = rest.pop(0)
        if x in rest:
            rest.remove(x)
            pairs.append((x, x))
        else:
            return False, pairs
    return True, pairs

print(pair_up([3, 2, 3, 2, 2, 2]))
print(pair_up([1, 2, 3, 4]))

(True, [(3, 3), (2, 2), (2, 2)])
(False, [])


In [9]:
from typing import List

class SolutionBrute:
    def divideArray(self, nums: List[int]) -> bool:
        rest = list(nums)
        while rest:
            x = rest.pop(0)
            if x in rest:
                rest.remove(x)
            else:
                return False
        return True

print(SolutionBrute().divideArray([3, 2, 3, 2, 2, 2]))
print(SolutionBrute().divideArray([1, 2, 3, 4]))

True
False


**Độ phức tạp:** `pop(0)`, `in`, `remove` đều `O(n)` và chạy `n` lần nên **Time `O(n²)`**, **Space `O(n)`** (bản sao). Với `2n ≤ 1000` vẫn đủ nhanh nhưng lãng phí.

**Nút thắt:** mỗi lần tìm bạn cặp ta quét lại danh sách. Biết trước "số lần xuất hiện" thì khỏi phải tìm.

## 6. Ý tưởng thuật toán / bước đột phá

Insight: **chia được thành các cặp bằng nhau ⇔ mọi giá trị có tần suất chẵn**. Chứng minh ngắn:

- nếu mọi tần suất chẵn, ta lấy từng giá trị `v` có tần suất `2k` và tạo `k` cặp `(v, v)`;
- nếu có giá trị `v` với tần suất lẻ, mỗi cặp chứa 0 hoặc 2 phần tử `v`, nên tổng số `v` được dùng luôn chẵn → không thể dùng hết các `v`.

Quy tắc: **nếu tồn tại giá trị có tần suất lẻ thì `False`, ngược lại `True`**. Chỉ cần đếm tần suất (hoặc theo dõi parity) trong `O(n)`.

## 7. Các cách tiếp theo

### Cách 2 — `Counter` + `all`

Đếm tần suất một lần rồi kiểm tra mọi tần suất đều chẵn.

In [10]:
from collections import Counter
from typing import List

class SolutionCounter:
    def divideArray(self, nums: List[int]) -> bool:
        cnt = Counter(nums)
        return all(c % 2 == 0 for c in cnt.values())

print(SolutionCounter().divideArray([3, 2, 3, 2, 2, 2]))
print(SolutionCounter().divideArray([1, 2, 3, 4]))

True
False


**Độ phức tạp:** Time `O(n)`, Space `O(k)` với `k` là số giá trị khác nhau (`≤ 500`).

### Cách 3 — Set bật/tắt (toggle)

Không cần lưu tần suất đầy đủ, chỉ cần **parity**: gặp lần lẻ thì thêm, gặp lần chẵn thì xóa. Cuối cùng set rỗng ⇔ mọi giá trị xuất hiện chẵn lần.

In [11]:
from typing import List

class SolutionSet:
    def divideArray(self, nums: List[int]) -> bool:
        odd = set()
        for x in nums:
            if x in odd:
                odd.remove(x)
            else:
                odd.add(x)
        return not odd

print(SolutionSet().divideArray([3, 2, 3, 2, 2, 2]))
print(SolutionSet().divideArray([1, 2, 3, 4]))

True
False


**Độ phức tạp:** Time `O(n)`, Space `O(k)` (trong xấu nhất set chứa tất cả giá trị lẻ). Ngắn và tiết kiệm bộ nhớ hơn `Counter` khi nhiều giá trị lặp.

### Cách 4 — Sắp xếp rồi so từng cặp kề nhau

Sau khi sắp xếp, các giá trị bằng nhau liền kề. Nếu chia được thì cặp `(a[0],a[1]), (a[2],a[3]), ...` phải **mỗi cặp hai số bằng nhau**. Ngược lại, nếu có `a[2i] != a[2i+1]` thì có giá trị bị lẻ ở đoạn đó.

In [12]:
from typing import List

class SolutionSort:
    def divideArray(self, nums: List[int]) -> bool:
        arr = sorted(nums)
        for i in range(0, len(arr), 2):
            if arr[i] != arr[i + 1]:
                return False
        return True

print(SolutionSort().divideArray([3, 2, 3, 2, 2, 2]))
print(SolutionSort().divideArray([1, 2, 3, 4]))

True
False


**Độ phức tạp:** Time `O(n log n)` do sắp xếp, Space `O(n)` (bản sao đã sắp xếp; `nums.sort()` tại chỗ thì `O(1)` phụ). Cách này dùng được dù không có dictionary.

### Cách 5 — Mảng đếm theo giá trị + XOR parity

Vì `1 ≤ nums[i] ≤ 500` ta dùng mảng `parity` dài 501: mỗi lần gặp `x` đảo `parity[x] ^= 1`. Cuối cùng chỉ cần `sum(parity) == 0` (tức mọi giá trị chẵn lần). `^ 1` đảo bit: `0 → 1`, `1 → 0`.

In [13]:
from typing import List

class SolutionParityArray:
    def divideArray(self, nums: List[int]) -> bool:
        parity = [0] * 501
        for x in nums:
            parity[x] ^= 1
        return sum(parity) == 0

print(SolutionParityArray().divideArray([3, 2, 3, 2, 2, 2]))
print(SolutionParityArray().divideArray([1, 2, 3, 4]))
print([0 ^ 1, 1 ^ 1, 1 ^ 0])

True
False
[1, 0, 1]


**Độ phức tạp:** Time `O(n + V)` với `V = 501` (hằng số), Space `O(V)`. Nhanh vì chỉ chỉ số mảng, không băm.

## 8. TRACE từng bước

Trace Cách 3 (set toggle) với `nums = [3,2,3,2,2,2]`:

| bước | `x` | `x in odd` trước | hành động | `odd` sau |
|---|---|---|---|---|
| 1 | 3 | không | thêm | {3} |
| 2 | 2 | không | thêm | {3, 2} |
| 3 | 3 | có | xóa | {2} |
| 4 | 2 | có | xóa | {} |
| 5 | 2 | không | thêm | {2} |
| 6 | 2 | có | xóa | {} |

`odd` rỗng → `True`.

In [14]:
def trace_set(nums):
    odd = set()
    for step, x in enumerate(nums, 1):
        before = x in odd
        if before:
            odd.remove(x)
            act = "xóa"
        else:
            odd.add(x)
            act = "thêm"
        print(f"bước {step}: x={x}, trước đó có trong set={before!s:5} -> {act}, odd={sorted(odd)}")
    return not odd

print("kết quả:", trace_set([3, 2, 3, 2, 2, 2]))
print("kết quả:", trace_set([1, 2, 3, 4]))

bước 1: x=3, trước đó có trong set=False -> thêm, odd=[3]
bước 2: x=2, trước đó có trong set=False -> thêm, odd=[2, 3]
bước 3: x=3, trước đó có trong set=True  -> xóa, odd=[2]
bước 4: x=2, trước đó có trong set=True  -> xóa, odd=[]
bước 5: x=2, trước đó có trong set=False -> thêm, odd=[2]
bước 6: x=2, trước đó có trong set=True  -> xóa, odd=[]
kết quả: True
bước 1: x=1, trước đó có trong set=False -> thêm, odd=[1]
bước 2: x=2, trước đó có trong set=False -> thêm, odd=[1, 2]
bước 3: x=3, trước đó có trong set=False -> thêm, odd=[1, 2, 3]
bước 4: x=4, trước đó có trong set=False -> thêm, odd=[1, 2, 3, 4]
kết quả: False


In [15]:
def trace_sort(nums):
    arr = sorted(nums)
    print("đã sắp xếp:", arr)
    for i in range(0, len(arr), 2):
        ok = arr[i] == arr[i + 1]
        print(f"  cặp ({i},{i+1}): {arr[i]} vs {arr[i+1]} -> {'khớp' if ok else 'LỆCH'}")
        if not ok:
            return False
    return True

print(trace_sort([3, 2, 3, 2, 2, 2]))
print(trace_sort([1, 2, 3, 4]))

đã sắp xếp: [2, 2, 2, 2, 3, 3]
  cặp (0,1): 2 vs 2 -> khớp
  cặp (2,3): 2 vs 2 -> khớp
  cặp (4,5): 3 vs 3 -> khớp
True
đã sắp xếp: [1, 2, 3, 4]
  cặp (0,1): 1 vs 2 -> LỆCH
False


In [16]:
def trace_parity_array(nums):
    parity = [0] * 6          # demo với giá trị nhỏ 0..5 cho dễ nhìn
    for step, x in enumerate(nums, 1):
        parity[x] ^= 1
        print(f"bước {step}: x={x} -> parity={parity}")
    print("tổng parity =", sum(parity), "->", "chia được" if sum(parity) == 0 else "không chia được")

trace_parity_array([3, 2, 3, 2, 2, 2])
trace_parity_array([1, 2, 3, 4])

bước 1: x=3 -> parity=[0, 0, 0, 1, 0, 0]
bước 2: x=2 -> parity=[0, 0, 1, 1, 0, 0]
bước 3: x=3 -> parity=[0, 0, 1, 0, 0, 0]
bước 4: x=2 -> parity=[0, 0, 0, 0, 0, 0]
bước 5: x=2 -> parity=[0, 0, 1, 0, 0, 0]
bước 6: x=2 -> parity=[0, 0, 0, 0, 0, 0]
tổng parity = 0 -> chia được
bước 1: x=1 -> parity=[0, 1, 0, 0, 0, 0]
bước 2: x=2 -> parity=[0, 1, 1, 0, 0, 0]
bước 3: x=3 -> parity=[0, 1, 1, 1, 0, 0]
bước 4: x=4 -> parity=[0, 1, 1, 1, 1, 0]
tổng parity = 4 -> không chia được


**Đọc bảng trace parity:** mỗi ô `parity[v]` là một "công tắc" cho giá trị `v`. Gặp `v` lần 1 bật (1), lần 2 tắt (0), lần 3 bật lại... Ở cuối, ô nào còn bật nghĩa là `v` xuất hiện lẻ lần. Ví dụ `[1,2,3,4]` kết thúc với 4 công tắc bật nên không chia được.

**Gợi ý cách tự kiểm tra bằng tay:** với mảng dài hơn, đừng ghép cặp từng phần tử; hãy gạch chéo từng giá trị khi gặp lần thứ hai (giống toggle). Phần tử nào còn lại không bị gạch chính là phần thừa.

## 9. Code nộp LeetCode

In [17]:
from collections import Counter
from typing import List

class Solution:
    def divideArray(self, nums: List[int]) -> bool:
        return all(c % 2 == 0 for c in Counter(nums).values())

print(Solution().divideArray([3, 2, 3, 2, 2, 2]))
print(Solution().divideArray([1, 2, 3, 4]))

True
False


## 10. Vì sao thuật toán đúng

- **Điều kiện cần:** mỗi cặp hợp lệ chứa hai số **cùng giá trị**. Với một giá trị `v`, số phần tử bằng `v` được dùng trong các cặp là `2 × (số cặp (v,v))` — luôn **chẵn**. Muốn dùng hết thì tần suất của `v` phải chẵn.
- **Điều kiện đủ:** nếu tần suất `v` là `2k`, ghép `k` cặp `(v, v)`; làm vậy cho mọi giá trị là một cách chia hợp lệ, mỗi phần tử dùng đúng một lần.
- **Cách toggle** giữ **invariant**: sau khi đọc một tiền tố của mảng, `odd` chứa đúng các giá trị có tần suất lẻ trong tiền tố đó (mỗi lần đọc `x` đảo parity của `x`). Cuối mảng, `odd` rỗng ⇔ mọi tần suất chẵn.
- **Cách sắp xếp:** nếu mọi tần suất chẵn thì mỗi nhóm bằng nhau có độ dài chẵn, nên các nhóm bắt đầu ở chỉ số chẵn, và `a[2i] == a[2i+1]`. Nếu có nhóm lẻ, lần đầu tiên một nhóm lẻ xuất hiện sẽ làm lệch phase ở vị trí chẵn, và sẽ bị phát hiện ở một cặp `(a[2i], a[2i+1])` khác giá trị.

## 11. Phân tích Time / Space Complexity

Đặt `N = len(nums)` (tức `2n`), `k` = số giá trị khác nhau (`k ≤ 500`).

| Cách | Time | Space |
|---|---|---|
| 1. Ghép trực tiếp (`pop`, `remove`) | `O(N²)` | `O(N)` |
| 2. Counter + all | `O(N)` | `O(k)` |
| 3. Set toggle | `O(N)` | `O(k)` |
| 4. Sắp xếp + so cặp | `O(N log N)` | `O(N)` |
| 5. Mảng parity (XOR) | `O(N + V)`, `V = 501` | `O(V)` |

Với `N ≤ 1000`, mọi cách đều chạy ngay; điểm khác nhau chỉ ở việc có rèn được tư duy "đếm parity" hay không.

## 12. Bộ test

Ví dụ đề, edge case (hai phần tử, toàn bộ bằng nhau, tần suất lẻ rải rác), phản ví dụ và test ngẫu nhiên có seed đối chiếu **tất cả cách**.

In [18]:
import random

solvers = [SolutionBrute(), SolutionCounter(), SolutionSet(), SolutionSort(), SolutionParityArray(), Solution()]

fixed = [
    ([3, 2, 3, 2, 2, 2], True),
    ([1, 2, 3, 4], False),
    ([1, 1], True),                       # nhỏ nhất, chia được
    ([1, 2], False),                      # nhỏ nhất, không chia được
    ([5] * 10, True),                     # tất cả bằng nhau
    ([1, 1, 1, 2, 2, 2], False),          # phản ví dụ: tổng chẵn nhưng mỗi tần suất lẻ
    ([500, 1, 500, 1], True),             # biên giá trị
    ([1, 2, 1, 3], False),
]
for nums, expected in fixed:
    for sol in solvers:
        assert sol.divideArray(list(nums)) == expected, (type(sol).__name__, nums)
print("test cố định: OK")

random.seed(2206)
for _ in range(500):
    n = random.randint(1, 8)
    nums = [random.randint(1, 4) for _ in range(2 * n)]
    expected = all(nums.count(v) % 2 == 0 for v in set(nums))
    for sol in solvers:
        assert sol.divideArray(list(nums)) == expected, (type(sol).__name__, nums)
# test chắc chắn True: tạo từ n cặp rồi xáo
for _ in range(100):
    n = random.randint(1, 20)
    nums = [random.randint(1, 500) for _ in range(n)] * 2
    random.shuffle(nums)
    assert all(sol.divideArray(list(nums)) for sol in solvers)
print("600 test ngẫu nhiên: OK")

test cố định: OK
600 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra "số giá trị khác nhau" hoặc độ dài

Độ dài luôn chẵn theo đề nên `len(nums) % 2 == 0` luôn `True` và **không cho thông tin gì** về việc các số có ghép cặp được hay không. Điều cần kiểm tra là **giá trị** của các phần tử.

In [19]:
def wrong1(nums):
    return len(nums) % 2 == 0                      # luôn True với đề này

print("sai:", wrong1([1, 2, 3, 4]), "(đáng lẽ False)")
print("đúng:", Solution().divideArray([1, 2, 3, 4]))

sai: True (đáng lẽ False)
đúng: False


### Lỗi 2: Kiểm tra tần suất `>= 2` thay vì **chẵn**

Một giá trị xuất hiện 3 lần vẫn "có cặp" nhưng còn dư một phần tử.

In [20]:
from collections import Counter

def wrong2(nums):
    return all(c >= 2 for c in Counter(nums).values())

nums = [1, 1, 1, 2, 2, 2]
print("sai:", wrong2(nums), "| đúng:", Solution().divideArray(nums))

sai: True | đúng: False


### Lỗi 3: Sắp xếp rồi so `a[i] == a[i+1]` với **mọi** i

Phải so theo bước 2 (`i = 0, 2, 4, ...`). So mọi `i` sẽ đòi cả `a[1] == a[2]`, tức bắt tất cả phần tử bằng nhau.

In [21]:
def wrong3(nums):
    a = sorted(nums)
    return all(a[i] == a[i + 1] for i in range(len(a) - 1))

print("sai:", wrong3([3, 2, 3, 2, 2, 2]), "(đáng lẽ True) | đúng:", Solution().divideArray([3, 2, 3, 2, 2, 2]))

sai: False (đáng lẽ True) | đúng: True


### Lỗi 4: Dùng `set.remove` khi phần tử không có trong set

`seen.remove(x)` báo `KeyError` nếu `x` không có. Dùng `discard` hoặc kiểm tra `in` trước.

In [22]:
s = set()
try:
    s.remove(5)
except KeyError as e:
    print("sai -> KeyError:", e)
s.discard(5)                 # an toàn: không lỗi
print("đúng:", s)

sai -> KeyError: 5
đúng: set()


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chia thành các cặp / nhóm bằng nhau" → nghĩ đến **tần suất** và **chia hết** (`% 2`, `% k`).
- Thấy "mọi giá trị xuất hiện chẵn lần" → có thể dùng **XOR** (với một giá trị), **toggle set**, hoặc mảng parity.
- Trên LeetCode: `all(c % 2 == 0 for c in Counter(nums).values())` là dòng thường dùng.
- Khi giá trị nhỏ (`≤ 500`), mảng đếm nhanh hơn dictionary và dễ viết bằng `[0] * 501`.
- Đừng sắp xếp nếu có thể đếm: sắp xếp `O(n log n)`, đếm `O(n)`.

## 15. Biến thể của bài

1. Chia mảng thành các nhóm **k phần tử bằng nhau** (mỗi tần suất chia hết cho `k`).
2. Tìm giá trị xuất hiện **lẻ lần** duy nhất (LeetCode 136 Single Number — XOR cả mảng).
3. Tính **số cặp tối đa** có thể tạo và số phần tử thừa (LeetCode 2341 Maximum Number of Pairs in Array).
4. Cặp là hai số **cộng bằng `k`** thay vì bằng nhau (LeetCode 1679 Max Number of K-Sum Pairs).
5. Cho phép **đổi** tối đa `m` phần tử để chia được: đếm số phần tử có tần suất lẻ rồi suy ra.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ghép trực tiếp | mô phỏng bằng tay, `remove` | `O(N²)` | `O(N)` | tự suy nghĩ ban đầu, N rất nhỏ |
| 2. Counter + all | mọi tần suất chẵn | `O(N)` | `O(k)` | **khuyến nghị khi nộp**, ngắn nhất |
| 3. Set toggle | giữ tập giá trị lẻ | `O(N)` | `O(k)` | nhớ ít, thể hiện tư duy parity |
| 4. Sắp xếp | so từng cặp kề nhau | `O(N log N)` | `O(N)` | không muốn dùng hash |
| 5. Mảng parity | `parity[x] ^= 1` | `O(N + V)` | `O(V)` | giá trị bị chặn nhỏ, cần tốc độ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `Counter([1,1,2]).values()` có bao nhiêu giá trị lẻ? Hàm `divideArray` cho `[1,1,2,2,2,2]` trả về gì?

**Bài 2 (tính tay).** Với `nums = [4,4,7,7,7,7,9,9,9,9,4,4]`, hãy đếm tần suất từng giá trị và kết luận chia được hay không.

**Bài 3 (điền TODO).** Hoàn thành hàm đếm **số phần tử thừa** nếu ghép cặp tối đa (mỗi giá trị bỏ lại `count % 2`).

In [23]:
def leftover(nums):
    # TODO: trả về số phần tử không ghép được khi ghép cặp tối đa
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `divide_into_groups(nums, k)` trả về `True` nếu chia được thành các nhóm `k` phần tử **bằng nhau** (không còn dư).

In [24]:
def divide_into_groups(nums, k):
    # TODO: tự viết
    pass

**Bài 5 (mini challenge).** Viết `single_odd_value(nums)` với giả định có **đúng một** giá trị xuất hiện lẻ lần, các giá trị khác xuất hiện chẵn lần. Trả về giá trị đó **chỉ dùng `O(1)` bộ nhớ phụ** (gợi ý: XOR).

In [25]:
def single_odd_value(nums):
    # TODO: dùng XOR
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: tần suất `1→2`, `2→4`: không có giá trị lẻ → `True`. Bài 2: đếm `4`, `7`, `9`.
- **Mức 2 (vừa):** Bài 3 cộng `c % 2` qua mọi tần suất của `Counter`. Bài 4: mọi tần suất phải chia hết cho `k`.
- **Mức 3 (sâu):** Bài 5: `a ^ a = 0` và `a ^ 0 = a`. XOR cả mảng, các cặp triệt tiêu nhau, chỉ còn giá trị lẻ.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
from collections import Counter

# Bài 1 và 2
assert sum(c % 2 for c in Counter([1, 1, 2]).values()) == 1
assert Solution().divideArray([1, 1, 2, 2, 2, 2]) is True
cnt = Counter([4, 4, 7, 7, 7, 7, 9, 9, 9, 9, 4, 4])
assert cnt == {4: 4, 7: 4, 9: 4}
assert Solution().divideArray([4, 4, 7, 7, 7, 7, 9, 9, 9, 9, 4, 4]) is True

# Bài 3
def leftover(nums):
    return sum(c % 2 for c in Counter(nums).values())

# Bài 4
def divide_into_groups(nums, k):
    return all(c % k == 0 for c in Counter(nums).values())

# Bài 5
def single_odd_value(nums):
    x = 0
    for v in nums:
        x ^= v
    return x

assert leftover([1, 1, 1, 2, 2, 2]) == 2 and leftover([1, 1, 2, 2]) == 0
assert divide_into_groups([1, 1, 1, 2, 2, 2], 3) is True
assert divide_into_groups([1, 1, 1, 2, 2], 3) is False
assert single_odd_value([4, 4, 7, 9, 9]) == 7
assert single_odd_value([5]) == 5
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mọi giá trị phải xuất hiện chẵn lần)

```python
from collections import Counter

class Solution:
    def divideArray(self, nums: List[int]) -> bool:
        return all(c % 2 == 0 for c in Counter(nums).values())
```

**Checklist**

- [ ] Đã chuyển đề thành điều kiện "mọi tần suất chẵn" (không chỉ `>= 2`)?
- [ ] Đã thử `[1,1,1,2,2,2]` (tần suất lẻ nhưng tổng chẵn) → `False`?
- [ ] Nếu sắp xếp: so theo bước 2, không so mọi cặp kề?
- [ ] Nếu dùng `set`: dùng `remove` chỉ sau khi kiểm tra `in`, hoặc dùng `discard`?
- [ ] Đã thử `[5]*10` (tất cả bằng nhau) và `[1,2]` (nhỏ nhất, sai)?
- [ ] Signature đúng `divideArray(self, nums: List[int]) -> bool` và có `from typing import List`?